<a href="https://colab.research.google.com/github/Allen-Vinicius/Pos_Estatistica_Probabilidade_Aplicada_Astronomia/blob/main/trabalho_pratico_em_R_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
suppressPackageStartupMessages({
  library(ggplot2) #pacote de visualização de dados.
  library(dplyr) # pacote de manipulação e transformação de dados
  install.packages("nortest") # Install the nortest package
  library(nortest) #pacote para testes para normalidade
})

# Helper functions (replace moments pkg)
skewness_fn <- function(x) {
  n <- length(x); m <- mean(x); s <- sd(x)
  n / ((n-1)*(n-2)) * sum(((x - m)/s)^3)
}
kurtosis_fn <- function(x) {
  n <- length(x); m <- mean(x); s <- sd(x) # Adicionado: Definição de n, m, s
  n*(n+1)/((n-1)*(n-2)*(n-3)) * sum(((x-m)/s)^4) - 3*(n-1)^2/((n-2)*(n-3))
}

# =============================================================================
# 1. SIMULAÇÃO DE DADOS
# =============================================================================
set.seed(42) #semente para reprodução dos dados
mu_teorico  <- 15.5
sigma_ruido <- 0.8
n_obs       <- 1000

observacoes <- rnorm(n = n_obs, mean = mu_teorico, sd = sigma_ruido)

# Injetar outliers (falhas instrumentais ~2%)
n_outliers  <- 20
idx_outliers <- sample(1:n_obs, n_outliers)
observacoes[idx_outliers] <- observacoes[idx_outliers] +
  sample(c(-1, 1), n_outliers, replace = TRUE) * runif(n_outliers, 1.2, 2.5)

cat("======================================================\n")
cat(" SIMULAÇÃO DE BRILHO — ESTRELA VARIÁVEL\n")
cat("======================================================\n")
cat(sprintf("   μ teórico  : %.4f mag\n", mu_teorico))
cat(sprintf("   σ ruído    : %.4f mag\n", sigma_ruido))
cat(sprintf("   N observ.  : %d\n", n_obs))
cat(sprintf("   Outliers   : %d injetados (%.1f%%)\n\n", n_outliers, 100*n_outliers/n_obs))

# =============================================================================
# 2. ESTATÍSTICAS DESCRITIVAS
# =============================================================================
media_amostral <- mean(observacoes)
mediana        <- median(observacoes)
dp_amostral    <- sd(observacoes)
coef_var       <- dp_amostral / media_amostral * 100
assimetria     <- skewness_fn(observacoes)
curtose        <- kurtosis_fn(observacoes)

cat("======================================================\n")
cat(" ESTATÍSTICAS DESCRITIVAS\n")
cat("======================================================\n")
cat(sprintf("   Média amostral  : %.4f mag\n", media_amostral))
cat(sprintf("   Mediana         : %.4f mag\n", mediana))
cat(sprintf("   Desvio padrão   : %.4f mag\n", dp_amostral))
cat(sprintf("   Coef. de Var.   : %.2f %%%%\n",  coef_var))
cat(sprintf("   Assimetria      : %.4f\n",      assimetria))
cat(sprintf("   Curtose (excess): %.4f\n\n",    curtose))

# =============================================================================
# 3. VISUALIZAÇÕES
# =============================================================================
df <- data.frame(magnitude = observacoes)

# --- Histograma + densidade teórica -----------------------------------------
p1 <- ggplot(df, aes(x = magnitude)) +
  geom_histogram(aes(y = after_stat(density)), bins = 40,
                 fill = "#1B4F72", color = "white", alpha = 0.85) +
  stat_function(fun = dnorm, args = list(mean = mu_teorico, sd = sigma_ruido),
                color = "#F39C12", linewidth = 1.3, linetype = "dashed") +
  geom_density(color = "#E74C3C", linewidth = 1.1) +
  geom_vline(xintercept = media_amostral, color = "#2ECC71",
             linewidth = 1.0, linetype = "dotdash") +
  geom_vline(xintercept = mu_teorico, color = "#F39C12",
             linewidth = 1.0, linetype = "dashed") +
  annotate("text", x = media_amostral + 0.13, y = 1.3,
           label = sprintf("x̄ = %.3f", media_amostral),
           color = "#2ECC71", size = 4, fontface = "bold") +
  annotate("text", x = mu_teorico - 0.15, y = 1.3,
           label = sprintf("μ = %.3f", mu_teorico),
           color = "#F39C12", size = 4, fontface = "bold") +
  labs(title    = "Distribuição das Medições de Brilho Estelar",
       subtitle = "1.000 observações · Histograma + KDE + Curva teórica Normal",
       x = "Magnitude Aparente (mag)",
       y = "Densidade de Probabilidade",
       caption = "Linha verde: média amostral  |  Linha laranja tracejada: μ teórico  |  Curva vermelha: KDE amostral") +
  theme_minimal(base_size = 12) +
  theme(plot.title    = element_text(face = "bold", size = 14, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5, color = "grey40"),
        plot.caption  = element_text(size = 8,  color = "grey50"))

ggsave("/mnt/user-data/outputs/fig1_histograma.png", plot = p1,
       width = 9, height = 5.5, dpi = 180, create.dir = TRUE)

# --- Boxplot + jitter com outliers ------------------------------------------
Q1 <- quantile(observacoes, 0.25); Q3 <- quantile(observacoes, 0.75)
IQR_val <- IQR(observacoes)
lim_inf <- Q1 - 1.5 * IQR_val; lim_sup <- Q3 + 1.5 * IQR_val

df <- df %>%
  mutate(outlier_flag = magnitude < lim_inf | magnitude > lim_sup,
         categoria    = ifelse(outlier_flag,
                               "Outlier (falha instrumental)", "Leitura normal"))

p2 <- ggplot(df, aes(x = "", y = magnitude)) +
  geom_boxplot(fill = "#1B4F72", color = "white", alpha = 0.7,
               outlier.shape = NA, width = 0.4) +
  geom_jitter(aes(color = categoria), width = 0.15, alpha = 0.4, size = 1.1) +
  scale_color_manual(values = c("Leitura normal" = "#85C1E9",
                                "Outlier (falha instrumental)" = "#E74C3C")) +
  geom_hline(yintercept = c(lim_inf, lim_sup),
             linetype = "dashed", color = "#F39C12", linewidth = 0.9) +
  annotate("text", x = 1.32, y = lim_inf - 0.06,
           label = sprintf("LI = %.2f", lim_inf), color = "#F39C12", size = 3.5) +
  annotate("text", x = 1.32, y = lim_sup + 0.06,
           label = sprintf("LS = %.2f", lim_sup), color = "#F39C12", size = 3.5) +
  labs(title    = "Boxplot — Detecção de Outliers (Regra de Tukey 1.5×IQR)",
       subtitle = "Pontos vermelhos = possíveis falhas instrumentais",
       x = NULL, y = "Magnitude Aparente (mag)", color = "Classificação",
       caption = "Linhas tracejadas: limites Q1−1.5·IQR  e  Q3+1.5·IQR") +
  theme_minimal(base_size = 12) +
  theme(plot.title    = element_text(face = "bold", size = 14, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5, color = "grey40"),
        plot.caption  = element_text(size = 8,  color = "grey50"),
        legend.position = "bottom")

ggsave("/mnt/user-data/outputs/fig2_boxplot.png", plot = p2,
       width = 7, height = 6, dpi = 180, create.dir = TRUE)

# --- QQ-Plot ----------------------------------------------------------------
p3 <- ggplot(df, aes(sample = magnitude)) +
  stat_qq(color = "#1B4F72", alpha = 0.5, size = 1.2) +
  stat_qq_line(color = "#E74C3C", linewidth = 1.2) +
  labs(title    = "Q-Q Plot — Verificação de Aderência à Normal",
       subtitle = "Desvios nas caudas indicam presença de outliers",
       x = "Quantis Teóricos (Normal Padrão)",
       y = "Quantis Amostrais (Magnitude)") +
  theme_minimal(base_size = 12) +
  theme(plot.title    = element_text(face = "bold", size = 14, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5, color = "grey40"))

ggsave("/mnt/user-data/outputs/fig3_qqplot.png", plot = p3,
       width = 7, height = 5.5, dpi = 180, create.dir = TRUE)

cat("  Gráficos gerados com sucesso.\n\n")

# =============================================================================
# 4. INTERVALO DE CONFIANÇA 95%
# =============================================================================
alpha       <- 0.05
t_crit      <- qt(1 - alpha/2, df = n_obs - 1)
erro_padrao <- dp_amostral / sqrt(n_obs)
IC_inf      <- media_amostral - t_crit * erro_padrao
IC_sup      <- media_amostral + t_crit * erro_padrao

cat("======================================================\n")
cat(" INTERVALO DE CONFIANÇA — 95% (t de Student)\n")
cat("======================================================\n")
cat(sprintf("   Erro padrão (SE)  : %.5f mag\n", erro_padrao))
cat(sprintf("   t crítico (α/2)   : %.4f\n",     t_crit))
cat(sprintf("   IC 95%%%%            : [%.4f ; %.4f] mag\n", IC_inf, IC_sup))
cat(sprintf("   μ teórico está %s do IC\n\n",
            ifelse(mu_teorico >= IC_inf & mu_teorico <= IC_sup,
                   "DENTRO", "FORA")))

# =============================================================================
# 5. TESTE DE HIPÓTESE (t-test bilateral)
# =============================================================================
resultado_teste <- t.test(observacoes, mu = mu_teorico,
                          alternative = "two.sided", conf.level = 0.95)
t_stat  <- as.numeric(resultado_teste$statistic)
p_valor <- resultado_teste$p.value

cat("======================================================\n")
cat(" TESTE DE HIPÓTESE — t-test bilateral\n")
cat("======================================================\n")
cat(sprintf("   H0: μ = %.2f  |  H1: μ ≠ %.2f\n", mu_teorico, mu_teorico))
cat(sprintf("   Estatística t     : %.4f\n", t_stat))
cat(sprintf("   p-valor           : %.6f\n", p_valor))
cat(sprintf("   Decisão (α=0.05)  : %s H0\n\n",
            ifelse(p_valor < 0.05, "REJEITAR", "NÃO REJEITAR")))

# =============================================================================
# 6. DIAGNÓSTICO DE OUTLIERS E IMPACTO
# =============================================================================
n_out_detect <- sum(df$outlier_flag)
obs_limpas   <- observacoes[!df$outlier_flag]
media_limpa  <- mean(obs_limpas)
dp_limpo     <- sd(obs_limpas)
ep_limpo     <- dp_limpo / sqrt(length(obs_limpas))
t_limpo      <- qt(0.975, df = length(obs_limpas) - 1)
IC_limpo_inf <- media_limpa - t_limpo * ep_limpo
IC_limpo_sup <- media_limpa + t_limpo * ep_limpo

cat("======================================================\n")
cat(" DIAGNÓSTICO DE OUTLIERS (Regra de Tukey: 1.5×IQR)\n")
cat("======================================================\n")
cat(sprintf("   Outliers detectados : %d (%.1f%%%% das obs.)\n",
            n_out_detect, 100*n_out_detect/n_obs))
cat(sprintf("   Limite inferior     : %.4f mag\n", lim_inf))
cat(sprintf("   Limite superior     : %.4f mag\n\n", lim_sup))
cat("   Comparativo COM vs SEM outliers:\n")
cat(sprintf("   %-30s %8s  %8s\n", "Métrica", "COM", "SEM"))
cat(sprintf("   %-30s %8.4f  %8.4f\n", "Média (mag)",        media_amostral, media_limpa))
cat(sprintf("   %-30s %8.4f  %8.4f\n", "Desvio padrão (mag)",dp_amostral,    dp_limpo))
cat(sprintf("   %-30s %8.4f  %8.4f\n", "Viés (x̄ - μ)",
            media_amostral - mu_teorico, media_limpa - mu_teorico))
cat(sprintf("   IC 95%%%% COM : [%.4f ; %.4f]\n", IC_inf, IC_sup))
cat(sprintf("   IC 95%%%% SEM : [%.4f ; %.4f]\n\n", IC_limpo_inf, IC_limpo_sup))

# =============================================================================
# 7. TESTES DE NORMALIDADE
# =============================================================================
lil <- nortest::lillie.test(observacoes)
ad  <- nortest::ad.test(observacoes)

cat("======================================================\n")
cat(" TESTES DE NORMALIDADE\n")
cat("======================================================\n")
cat(sprintf("   Lilliefors  D = %.4f | p = %.5f | %s\n",
            lil$statistic, lil$p.value,
            ifelse(lil$p.value < 0.05, "Rejeita Normal (α=0.05)",
                   "Não rejeita Normal (α=0.05)")))
cat(sprintf("   And.-Darling A = %.4f | p = %.5f | %s\n",
            ad$statistic, ad$p.value,
            ifelse(ad$p.value < 0.05, "Rejeita Normal (α=0.05)",
                   "Não rejeita Normal (α=0.05)")))
cat("\n   Nota: a rejeição é esperada com outliers injetados;\n")
cat("   confirma a sensibilidade dos testes à contaminação.\n\n")

cat("======================================================\n")
cat(" ANÁLISE CONCLUÍDA COM SUCESSO\n")
cat("======================================================\n")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



 SIMULAÇÃO DE BRILHO — ESTRELA VARIÁVEL
   μ teórico  : 15.5000 mag
   σ ruído    : 0.8000 mag
   N observ.  : 1000
   Outliers   : 20 injetados (2.0%)

 ESTATÍSTICAS DESCRITIVAS
   Média amostral  : 15.4713 mag
   Mediana         : 15.4823 mag
   Desvio padrão   : 0.8468 mag
   Coef. de Var.   : 5.47 %%
   Assimetria      : 0.0154
   Curtose (excess): 0.7616

  Gráficos gerados com sucesso.

 INTERVALO DE CONFIANÇA — 95% (t de Student)
   Erro padrão (SE)  : 0.02678 mag
   t crítico (α/2)   : 1.9623
   IC 95%%            : [15.4188 ; 15.5239] mag
   μ teórico está DENTRO do IC

 TESTE DE HIPÓTESE — t-test bilateral
   H0: μ = 15.50  |  H1: μ ≠ 15.50
   Estatística t     : -1.0702
   p-valor           : 0.284771
   Decisão (α=0.05)  : NÃO REJEITAR H0

 DIAGNÓSTICO DE OUTLIERS (Regra de Tukey: 1.5×IQR)
   Outliers detectados : 17 (1.7%% das obs.)
   Limite inferior     : 13.2977 mag
   Limite superior     : 17.6705 mag

   Comparativo COM vs SEM outliers:
   Métrica                     